In [ ]:
#%% md
# Ноутбук 4: Потоковая (инкрементальная) дедупликация vs батчевая обработка
#
# Гипотеза H: вместо обработки всего датасета сразу (как в базовом ноутбуке и ноутбуке 2)
#     эмулируем реальный Telegram-поток — обрабатываем новости по одной, в порядке времени,
#     поддерживая скользящее окно активных векторов (только последние WINDOW_DAYS дней).
#     Для новой новости сразу считаем похожесть только с активным окном (не со всем корпусом)
#     и сразу решаем, дубль это или нет — как это работало бы в проде в реальном времени.
#
# НЕ включаем сюда сравнение моделей эмбеддинга (ноутбук 2) — намеренно используем один и тот
# же TF-IDF (та же векторизация, что и batch-референс), чтобы разница в результатах была
# ТОЛЬКО из-за потоковой vs батчевой архитектуры, а не из-за смены модели.
#%%


In [ ]:
from pathlib import Path
import re, time
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

DATA_PATH = '/kaggle/input/datasets/artemwalker/telegram-news/telegram_news_raw.csv'
OUT_DIR = Path('streaming_dedup_results')
OUT_DIR.mkdir(exist_ok=True)
WINDOW_DAYS = 14
THRESHOLD = 0.90
N_NEIGHBORS = 6

df = pd.read_csv(DATA_PATH, parse_dates=['post_dttm']).reset_index(drop=True)

def normalize_text(text):
    text = str(text or '').lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@[\w_]+', ' ', text)
    text = re.sub(r'[^\w\s%.-]', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()

df['text_normalized'] = df['data'].fillna('').map(normalize_text)
df = df[df.text_normalized.str.len().gt(0)].copy().reset_index(drop=True)
df = df.sort_values('post_dttm').reset_index(drop=True)  # поток строго по времени
print(f'Новостей: {len(df):,}')

class UnionFind:
    def __init__(self, n): self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b: self.parent[b] = a

def clusters_from_edges(n, edges):
    uf = UnionFind(n)
    for a, b in edges: uf.union(a, b)
    return pd.factorize([uf.find(i) for i in range(n)])[0]

def top_k_edges(vectors, dates, threshold, n_neighbors=6, batch_size=128):
    ts = pd.to_datetime(dates, utc=True).astype('int64').to_numpy()
    max_delta = pd.Timedelta(days=WINDOW_DAYS).value
    edges = []
    for start in range(0, len(ts), batch_size):
        stop = min(start + batch_size, len(ts))
        sim = vectors[start:stop] @ vectors.T
        if hasattr(sim, 'toarray'): sim = sim.toarray()
        idx = np.argpartition(sim, -n_neighbors, axis=1)[:, -n_neighbors:]
        for local_i, candidates in enumerate(idx):
            i = start + local_i
            for j in candidates:
                if i < j and sim[local_i, j] >= threshold and abs(ts[i] - ts[j]) <= max_delta:
                    edges.append((i, int(j)))
    return edges

tfidf = TfidfVectorizer(analyzer='word', ngram_range=(2, 5), min_df=4, max_features=500_000)
tfidf_vectors = tfidf.fit_transform(df.text_normalized)

# --- Батчевый референс (как в базовом ноутбуке) ---
t0 = time.time()
batch_edges = top_k_edges(tfidf_vectors, df.post_dttm, THRESHOLD, n_neighbors=N_NEIGHBORS)
batch_clusters = clusters_from_edges(len(df), batch_edges)
batch_time = time.time() - t0
print(f'Батч: {len(batch_edges)} пар за {batch_time:.1f} с')

In [ ]:
#%% md
### Гипотеза H: потоковая обработка со скользящим окном
# Как применяем: идём по новостям строго в хронологическом порядке. На каждом шаге держим
#     активное окно — индексы новостей за последние WINDOW_DAYS от текущей. Новую новость
#     сравниваем только с активным окном (через разреженное матричное умножение), находим
#     похожих (>= THRESHOLD), объединяем в Union-Find. Точно так же, как top_k_edges, но
#     инкрементально — без доступа к "будущим" новостям и без полного пересчёта.
# Как оцениваем результат:
#   - Согласие с батчевым результатом (ARI, NMI) — насколько потоковая версия близка к "идеалу".
#   - Время обработки на новость (должно быть примерно постоянным благодаря обрезке окна).
#   - Размер активного окна во времени (прокси памяти/вычислений в проде).
#%%
uf_stream = UnionFind(len(df))
dates_ns = df.post_dttm.astype('int64').to_numpy()
max_delta_ns = pd.Timedelta(days=WINDOW_DAYS).value

active_idx = []       # индексы новостей в текущем окне
per_message_time = []
window_size_over_time = []
stream_edges = []

for i in tqdm(range(len(df)), desc='Потоковая обработка'):
    t_start = time.perf_counter()

    # обрезаем окно: убираем новости старше WINDOW_DAYS от текущей
    cutoff = dates_ns[i] - max_delta_ns
    active_idx = [j for j in active_idx if dates_ns[j] >= cutoff]

    if active_idx:
        sub = tfidf_vectors[active_idx]
        sims = (tfidf_vectors[i] @ sub.T).toarray().ravel()
        for local_j, sim in enumerate(sims):
            if sim >= THRESHOLD:
                j = active_idx[local_j]
                uf_stream.union(i, j)
                stream_edges.append((min(i, j), max(i, j)))

    active_idx.append(i)
    per_message_time.append(time.perf_counter() - t_start)
    window_size_over_time.append(len(active_idx))

stream_clusters = pd.factorize([uf_stream.find(i) for i in range(len(df))])[0]
total_stream_time = sum(per_message_time)
print(f'Поток: {len(stream_edges)} пар за {total_stream_time:.1f} с '
      f'({np.mean(per_message_time)*1000:.2f} мс/новость в среднем)')

In [ ]:
#%% md
## Оценка гипотезы H: поток vs батч
#%%
ari = adjusted_rand_score(batch_clusters, stream_clusters)
nmi = normalized_mutual_info_score(batch_clusters, stream_clusters)
print(f'ARI (поток vs батч) = {ari:.3f}')
print(f'NMI (поток vs батч) = {nmi:.3f}')
print(f'Батч время: {batch_time:.1f} с; поток суммарное время: {total_stream_time:.1f} с')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(np.array(per_message_time) * 1000, bins=50)
axes[0].set_title('Время обработки одной новости в потоке')
axes[0].set_xlabel('мс')
axes[0].set_ylabel('Число новостей')

axes[1].plot(window_size_over_time)
axes[1].set_title('Размер активного окна во времени (прокси памяти)')
axes[1].set_xlabel('Порядковый номер новости в потоке')
axes[1].set_ylabel('Новостей в окне')
plt.tight_layout()
plt.savefig(OUT_DIR / 'streaming_vs_batch.png', dpi=150)
plt.show()

pd.DataFrame({
    'metric': ['ARI', 'NMI', 'batch_time_sec', 'stream_time_sec', 'avg_ms_per_message'],
    'value': [ari, nmi, batch_time, total_stream_time, np.mean(per_message_time) * 1000]
}).to_csv(OUT_DIR / 'streaming_evaluation.csv', index=False, encoding='utf-8-sig')

In [ ]:
# Итог ноутбука 4:
#  - streaming_evaluation.csv — ARI/NMI поток vs батч, суммарное время обеих архитектур (гипотеза H)
#  - streaming_vs_batch.png — распределение задержки на сообщение и рост активного окна
print('Готово. Файлы результатов сохранены в', OUT_DIR.resolve())